# Chapitre 16 - Cours 2 : apprendre un ensemble, puis le post-traiter

Ce second notebook couvre la section 16.3 du livre. L'idée, due à Friedman et
Popescu, est de **séparer** les deux tâches de l'apprentissage ensembliste :

1. construire un dictionnaire **fini** de fonctions de base (des arbres) à partir des
   données ;
2. les **combiner** en ajustant un chemin de lasso dans ce dictionnaire.

Nous verrons le post-traitement d'une forêt aléatoire sur les données spam
(figure 16.6), l'algorithme ISLE de génération d'ensembles (algorithme 16.2,
figure 16.8) et les **ensembles de règles** (RuleFit simplifié, figure 16.10).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch16_ensembles import (standardize, lasso_path_cd, lasso_path_fast,
                                isle_ensemble, tree_predictions,
                                staged_ensemble_predictions, extract_rules,
                                rule_matrix, rule_to_str)

## 1. Le modèle et le post-traitement lasso

On considère des fonctions de la forme (éq. (16.8) du livre)
$$
f(x)=\alpha_0+\sum_{T_k\in\mathcal T}\alpha_k\,T_k(x),
$$
où $\mathcal T$ est un dictionnaire de fonctions de base, typiquement des arbres. Le
cours 1 a montré que le gradient boosting avec rétrécissement suit un chemin
régularisé $L_1$ monotone dans l'espace de **tous** les arbres. Friedman et Popescu
proposent une approche hybride en deux étapes :

- induire à partir des données un dictionnaire fini
  $\mathcal T_L=\{T_1(x),\dots,T_M(x)\}$ ;
- ajuster un chemin de lasso dans ce dictionnaire (éq. (16.9)) :
$$
\alpha(\lambda)=\arg\min_\alpha\ \sum_{i=1}^N L\Big[y_i,\ \alpha_0+\sum_{m=1}^M\alpha_mT_m(x_i)\Big]+\lambda\sum_{m=1}^M|\alpha_m|.
$$

> **Intuition.** Dans sa forme la plus simple, c'est un **post-traitement** d'une forêt
> aléatoire ou d'un modèle boosté : au lieu de faire la moyenne (ou la somme) de
> *tous* les arbres avec des poids fixés, on ré-estime leurs poids par un lasso. Le
> lasso en annule beaucoup : le modèle final est plus petit (moins de calculs et de
> stockage pour prédire), et souvent au moins aussi précis.

Concrètement : on calcule la matrice $N\times M$ des prédictions $T_m(x_i)$ de chaque
arbre sur l'échantillon d'apprentissage, puis on la traite comme une matrice de
prédicteurs ordinaire pour le lasso. Nous utilisons la perte quadratique
$L(y,f)=(y-f)^2$, y compris en classification (réponse codée 0/1, on prédit la
classe 1 si $\hat f(x)>1/2$).

## 2. Post-traitement d'une forêt aléatoire sur les données spam (figure 16.6)

Les données spam (4601 courriels, 57 prédicteurs) n'ont pas de découpage
apprentissage/test fourni ici ; nous tirons au hasard 1536 courriels de test, comme
dans le livre.

In [2]:
spam = datasets.load_spam()
Xs_all = spam.drop(columns="spam").to_numpy(float)
ys_all = spam["spam"].to_numpy(int)
rng = np.random.RandomState(0)
perm = rng.permutation(len(ys_all))
te, tr = perm[:1536], perm[1536:]
Xs, ys, Xs_te, ys_te = Xs_all[tr], ys_all[tr], Xs_all[te], ys_all[te]
print("apprentissage :", Xs.shape, " test :", Xs_te.shape)

apprentissage : (3065, 57)  test : (1536, 57)


**La forêt aléatoire.** 500 arbres de profondeur maximale, $m=7$ variables candidates
à chaque coupure (algorithme 15.1). Les forêts aléatoires ont été programmées au
chapitre 15 ; nous utilisons ici celles de scikit-learn pour la rapidité.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=500, max_features=7, random_state=0, n_jobs=4)
rf.fit(Xs, ys)
err_rf = np.mean(rf.predict(Xs_te) != ys_te)
print("erreur test de la forêt (500 arbres) : %.4f" % err_rf)